In [38]:
#Configuracion de metadatos y conexion
NOMBRE_GRUPO = "G3_FinanzasyMercado"
CATEGORIA_ACTUAL = "Finanzas"

In [47]:
import time
from pymongo import MongoClient
from selenium import webdriver
from selenium.webdriver.common.by import By
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Analisis_Categorias") \
    .config("spark.mongodb.read.connection.uri", "mongodb://database:27017/proyecto_bigdata") \
    .config("spark.mongodb.write.connection.uri", "mongodb://database:27017/proyecto_bigdata") \
    .config("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.12:10.6.1") \
    .getOrCreate()

# Aquí especificas explícitamente la colección
df = spark.read.format("mongodb") \
    .option("collection", "datos_scrapingG3") \
    .load()

print(df.count())   # debería dar 60
print(df.columns)   # confirma estructura

1000
['_id', 'categoria', 'fecha_captura', 'grupo', 'item', 'valor']


In [48]:
1 # Esto nos dice que columnas entendio Spark que habia en Mongo
df.printSchema()
# Esto nos muestra las primeras 5 filas para confirmar que hay datos reales
df.show(5)

root
 |-- _id: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- fecha_captura: string (nullable = true)
 |-- grupo: string (nullable = true)
 |-- item: string (nullable = true)
 |-- valor: double (nullable = true)

+--------------------+---------+-------------------+-------------------+--------------------+-----+
|                 _id|categoria|      fecha_captura|              grupo|                item|valor|
+--------------------+---------+-------------------+-------------------+--------------------+-----+
|69dd71cb29f6cc197...|   Travel|2026-04-13 22:43:04|G3_FinanzasyMercado|It's Only the Him...|45.17|
|69dd71cb29f6cc197...|   Travel|2026-04-13 22:43:04|G3_FinanzasyMercado|Full Moon over No...|49.43|
|69dd71cb29f6cc197...|   Travel|2026-04-13 22:43:04|G3_FinanzasyMercado|See America: A Ce...|48.87|
|69dd71cb29f6cc197...|   Travel|2026-04-13 22:43:04|G3_FinanzasyMercado|Vagabonding: An U...|36.94|
|69dd71cb29f6cc197...|   Travel|2026-04-13 22:43:05|G3_Finanzas

In [52]:
from pyspark.sql.functions import col

# Solo queremos registros que tengan un item real y un valor mayor a 0
df_filtrado = df.filter((col("item").isNotNull()) & (col("valor") > 0))

print("PASO 2: Limpieza completada.")
print("Registros originales:", df.count())
print("Registros validos:", df_filtrado.count())

PASO 2: Limpieza completada.
Registros originales: 1000
Registros validos: 1000


In [53]:
df.select("item", "valor").show()

+--------------------+-----+
|                item|valor|
+--------------------+-----+
|It's Only the Him...|45.17|
|Full Moon over No...|49.43|
|See America: A Ce...|48.87|
|Vagabonding: An U...|36.94|
|Under the Tuscan Sun|37.33|
|  A Summer In Europe|44.34|
|The Great Railway...|30.54|
|A Year in Provenc...|56.88|
|The Road to Littl...|23.21|
|Neither Here nor ...|38.95|
|1,000 Places to S...|26.08|
|       Sharp Objects|47.82|
|In a Dark, Dark Wood|19.63|
| The Past Never Ends| 56.5|
|    A Murder in Time|16.64|
|The Murder of Rog...| 44.1|
|The Last Mile (Am...|54.21|
|That Darkness (Ga...|13.92|
|Tastes Like Fear ...|10.69|
|A Time of Torment...|48.35|
+--------------------+-----+
only showing top 20 rows



In [54]:
df.filter(df["valor"] > 50).show()

+--------------------+------------------+-------------------+-------------------+--------------------+-----+
|                 _id|         categoria|      fecha_captura|              grupo|                item|valor|
+--------------------+------------------+-------------------+-------------------+--------------------+-----+
|69dd71cb29f6cc197...|            Travel|2026-04-13 22:43:05|G3_FinanzasyMercado|A Year in Provenc...|56.88|
|69dd71cb29f6cc197...|           Mystery|2026-04-13 22:43:06|G3_FinanzasyMercado| The Past Never Ends| 56.5|
|69dd71cb29f6cc197...|           Mystery|2026-04-13 22:43:06|G3_FinanzasyMercado|The Last Mile (Am...|54.21|
|69dd71cb29f6cc197...|           Mystery|2026-04-13 22:43:06|G3_FinanzasyMercado|Murder at the 42n...|54.36|
|69dd71cb29f6cc197...|           Mystery|2026-04-13 22:43:06|G3_FinanzasyMercado|Boar Island (Anna...|59.48|
|69dd71cb29f6cc197...|           Mystery|2026-04-13 22:43:06|G3_FinanzasyMercado|The Bachelor Girl...| 52.3|
|69dd71cb29f6cc197.

In [50]:
df.groupBy("categoria").count().show()

+--------------+-----+
|     categoria|count|
+--------------+-----+
|    Philosophy|   11|
|         Crime|    1|
|       Romance|   35|
|       Default|  152|
| Short Stories|    1|
|       Science|   14|
|      Thriller|   11|
|      Cultural|    1|
|     Christian|    3|
| Adult Fiction|    1|
|           Art|    8|
|Womens Fiction|   17|
|        Poetry|   19|
|    Nonfiction|  110|
|        Travel|   11|
|        Health|    4|
|      Academic|    1|
|       Erotica|    1|
|       Fiction|   65|
|      Religion|    7|
+--------------+-----+
only showing top 20 rows



In [55]:
# 2. Transformacion y Agregacion por CATEGORIA
# Esto permite comparar que generos son mas caros en promedio
reporte_categorias = df.groupBy("categoria").agg(
    F.count("item").alias("cantidad_libros"),
    F.avg("valor").alias("precio_promedio"),
    F.min("valor").alias("precio_minimo"),
    F.max("valor").alias("precio_maximo")
).orderBy(F.desc("precio_promedio"))

print("ANALISIS DE MERCADO POR CATEGORIA:")
reporte_categorias.show()

ANALISIS DE MERCADO POR CATEGORIA:
+----------------+---------------+------------------+-------------+-------------+
|       categoria|cantidad_libros|   precio_promedio|precio_minimo|precio_maximo|
+----------------+---------------+------------------+-------------+-------------+
|        Suspense|              1|             58.33|        58.33|        58.33|
|          Novels|              1|             54.81|        54.81|        54.81|
|        Politics|              3| 53.61333333333332|        51.33|        56.86|
|          Health|              4|           51.4525|        49.05|        54.07|
|       New Adult|              6| 46.38333333333333|        36.29|        58.35|
|       Christian|              3| 42.49666666666666|        25.77|         54.0|
|Sports and Games|              5|            41.166|        24.42|        51.22|
|       Self Help|              5|40.620000000000005|        12.08|        52.15|
|          Travel|             11| 39.79454545454546|        23

In [58]:
fila = df.orderBy(F.desc("valor")).select(
    "item", "grupo", "fecha_captura", "valor", "categoria"
).first()

print("Producto más caro:", fila["item"])
print("Categoría:", fila["categoria"])
print("Fecha y hora de captura:", fila["fecha_captura"])
print("Valor:", fila["valor"])

Producto más caro: The Perfect Play (Play by Play #1)
Categoría: Romance
Fecha y hora de captura: 2026-04-13 22:43:18
Valor: 59.99


In [64]:
df = df.withColumn("valor", F.regexp_replace("valor", "[^0-9.]", "").cast("double"))

In [66]:
df.select("categoria").distinct().show()

+--------------+
|     categoria|
+--------------+
|    Philosophy|
|         Crime|
|       Romance|
|       Default|
| Short Stories|
|       Science|
|      Thriller|
|      Cultural|
|     Christian|
| Adult Fiction|
|           Art|
|Womens Fiction|
|        Poetry|
|    Nonfiction|
|        Travel|
|        Health|
|      Academic|
|       Erotica|
|       Fiction|
|      Religion|
+--------------+
only showing top 20 rows



In [61]:
df.groupBy("categoria").count().show()

+--------------+-----+
|     categoria|count|
+--------------+-----+
|    Philosophy|   11|
|         Crime|    1|
|       Romance|   35|
|       Default|  152|
| Short Stories|    1|
|       Science|   14|
|      Thriller|   11|
|      Cultural|    1|
|     Christian|    3|
| Adult Fiction|    1|
|           Art|    8|
|Womens Fiction|   17|
|        Poetry|   19|
|    Nonfiction|  110|
|        Travel|   11|
|        Health|    4|
|      Academic|    1|
|       Erotica|    1|
|       Fiction|   65|
|      Religion|    7|
+--------------+-----+
only showing top 20 rows



In [70]:
df = df.withColumn("valor", F.regexp_replace("valor", "[^0-9,]", "").cast("double"))

In [72]:
df.select("valor").show(10)
df.printSchema()

+------+
| valor|
+------+
|4517.0|
|4943.0|
|4887.0|
|3694.0|
|3733.0|
|4434.0|
|3054.0|
|5688.0|
|2321.0|
|3895.0|
+------+
only showing top 10 rows

root
 |-- _id: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- fecha_captura: string (nullable = true)
 |-- grupo: string (nullable = true)
 |-- item: string (nullable = true)
 |-- valor: double (nullable = true)



In [82]:
from pyspark.sql import functions as F

ticket_salida = df.filter(F.col("grupo") == "G3_FinanzasyMercado") \
    .groupby("categoria") \
    .agg(
        F.count("item").alias("total_libros"),
        F.round(F.avg("valor"), 2).alias("precio_medio"),
        F.max("fecha_captura").alias("ultima_sincronizacion")
    )

print(f"--- TICKET DE SALIDA: {'G3_FinanzasyMercado'} ---")
ticket_salida.show(truncate=False)


--- TICKET DE SALIDA: G3_FinanzasyMercadoortega ---
+---------+------------+------------+---------------------+
|categoria|total_libros|precio_medio|ultima_sincronizacion|
+---------+------------+------------+---------------------+
+---------+------------+------------+---------------------+



In [60]:
print(df.columns)


['_id', 'categoria', 'fecha_captura', 'grupo', 'item', 'valor']
